# COVID-19 Global Employment Impact Analysis 🌍💼

## Comprehensive Analysis of Employment Losses and Recovery Patterns

This notebook provides an in-depth analysis of how COVID-19 impacted global employment patterns, working hours, and economic recovery across different countries, regions, and income levels. We'll explore employment data to understand:

- **Regional Employment Impact**: How different regions were affected
- **Gender Employment Disparities**: Male vs female employment changes
- **Economic Recovery Patterns**: Which countries showed resilience
- **Labor Market Vulnerability**: Dependency ratios and employment risks
- **Predictive Insights**: Models for employment recovery forecasting

---

### 📊 Key Metrics Analyzed:
- Total weekly hours worked (estimates in thousands)
- Percentage of working hours lost due to COVID-19
- Employment rates by gender (25+ age group)
- Labor dependency ratios
- Regional and income-level comparisons

### 🎯 Research Questions:
1. Which countries and regions experienced the highest employment losses?
2. How did COVID-19 impact employment differently across genders?
3. What factors correlate with employment vulnerability during crises?
4. Can we predict employment recovery patterns based on key indicators?

In [ ]:
# Import Required Libraries and Load Data
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.figure_factory as ff
import warnings
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler
import scipy.stats as stats

# Configure plotting settings
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 12
sns.set_style("whitegrid")
warnings.filterwarnings('ignore')

# Configure Plotly for inline display
import plotly.io as pio
pio.renderers.default = "plotly_mimetype+notebook"

print("📚 All libraries imported successfully!")
print("🔧 Plotting configurations set!")
print("📊 Ready for COVID-19 employment impact analysis!")

In [ ]:
# Load the employment data
try:
    # Update this path to match your data location
    df = pd.read_csv('c:\\Users\\ASUS\\Downloads\\CS661_Project\\employment_data.csv')
    print("✅ Employment data loaded successfully!")
    print(f"📊 Dataset shape: {df.shape[0]} rows × {df.shape[1]} columns")
except FileNotFoundError:
    print("❌ File not found. Please update the file path.")
    # For demo purposes, we'll create sample data structure
    print("Creating demo data structure...")

# Display basic dataset information
print("\n🔍 Dataset Overview:")
print("=" * 50)
print(f"Total countries/regions: {df.shape[0]}")
print(f"Total features: {df.shape[1]}")
print(f"Memory usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

# Display first few rows
print("\n📋 First 5 rows:")
df.head()

## 🔍 Data Exploration and Cleaning

Let's explore the dataset structure, identify data quality issues, and prepare the data for analysis.

In [ ]:
# Examine dataset structure and data types
print("📋 Column Information:")
print("=" * 50)
print(f"Columns: {list(df.columns)}")
print(f"\n📊 Data Types:")
print(df.dtypes)

print(f"\n🔍 Missing Values Analysis:")
missing_data = df.isnull().sum()
missing_percentage = (missing_data / len(df)) * 100
missing_df = pd.DataFrame({
    'Missing_Count': missing_data,
    'Missing_Percentage': missing_percentage
}).sort_values('Missing_Percentage', ascending=False)

print(missing_df[missing_df['Missing_Count'] > 0])

# Clean column names for easier handling
df.columns = [col.replace('(', '').replace(')', '').replace(' ', '_').replace('-', '_').replace(',', '') for col in df.columns]
print(f"\n✅ Column names cleaned for analysis")

# Display basic statistics
print(f"\n📈 Basic Statistics:")
df.describe()

In [ ]:
# Data cleaning and preparation
def clean_and_prepare_data(df):
    """Clean and prepare the employment data for analysis"""
    
    # Create a copy to avoid modifying original data
    df_clean = df.copy()
    
    # Remove regional/continental aggregates to focus on individual countries
    # Keep only actual countries (rows without income level or regional indicators)
    country_indicators = ['World', 'Africa', 'Americas', 'Asia', 'Europe', 'Arab', 'BRICS', 'G20', 'G7', 'ASEAN', 'MENA']
    df_countries = df_clean[~df_clean['country'].str.contains('|'.join(country_indicators), na=False)]
    df_countries = df_countries[~df_countries['country'].str.contains('income|Union|League|CARICOM', na=False)]
    
    # Create regional classification based on country names and existing patterns
    def assign_region(country):
        # African countries
        african_countries = ['Angola', 'Algeria', 'Botswana', 'Benin', 'Burkina Faso', 'Burundi', 'Cameroon', 
                           'Cape Verde', 'Central African Republic', 'Chad', 'Comoros', 'Congo', 
                           'Côte d\'Ivoire', 'Djibouti', 'Egypt', 'Equatorial Guinea', 'Eritrea', 
                           'Eswatini', 'Ethiopia', 'Gabon', 'Gambia', 'Ghana', 'Guinea', 'Guinea-Bissau',
                           'Kenya', 'Lesotho', 'Liberia', 'Libya', 'Madagascar', 'Malawi', 'Mali',
                           'Mauritania', 'Mauritius', 'Morocco', 'Mozambique', 'Namibia', 'Niger',
                           'Nigeria', 'Rwanda', 'Sao Tome and Principe', 'Senegal', 'Sierra Leone',
                           'Somalia', 'South Africa', 'South Sudan', 'Sudan', 'Tanzania', 'Togo',
                           'Tunisia', 'Uganda', 'Western Sahara', 'Zambia', 'Zimbabwe']
        
        # Asian countries
        asian_countries = ['Afghanistan', 'Bangladesh', 'Bhutan', 'Brunei Darussalam', 'Cambodia', 'China',
                          'India', 'Indonesia', 'Iran', 'Iraq', 'Japan', 'Jordan', 'Kazakhstan',
                          'Korea', 'Kuwait', 'Kyrgyzstan', 'Lao', 'Lebanon', 'Malaysia', 'Maldives',
                          'Mongolia', 'Myanmar', 'Nepal', 'Oman', 'Pakistan', 'Philippines', 'Qatar',
                          'Saudi Arabia', 'Singapore', 'Sri Lanka', 'Syrian Arab Republic', 'Tajikistan',
                          'Thailand', 'Timor-Leste', 'Turkey', 'Turkmenistan', 'United Arab Emirates',
                          'Uzbekistan', 'Viet Nam', 'Yemen', 'Azerbaijan', 'Bahrain', 'Taiwan', 'Hong Kong', 'Macau']
        
        # European countries
        european_countries = ['Albania', 'Armenia', 'Austria', 'Belarus', 'Belgium', 'Bosnia and Herzegovina',
                            'Bulgaria', 'Croatia', 'Cyprus', 'Czechia', 'Denmark', 'Estonia', 'Finland',
                            'France', 'Georgia', 'Germany', 'Greece', 'Hungary', 'Iceland', 'Ireland',
                            'Italy', 'Latvia', 'Lithuania', 'Luxembourg', 'Malta', 'Moldova', 'Montenegro',
                            'Netherlands', 'North Macedonia', 'Norway', 'Poland', 'Portugal', 'Romania',
                            'Russian Federation', 'Serbia', 'Slovakia', 'Slovenia', 'Spain', 'Sweden',
                            'Switzerland', 'Ukraine', 'United Kingdom', 'Channel Islands']
        
        # American countries
        american_countries = ['Argentina', 'Barbados', 'Belize', 'Bolivia', 'Brazil', 'Canada', 'Chile',
                            'Colombia', 'Costa Rica', 'Cuba', 'Dominican Republic', 'Ecuador', 'El Salvador',
                            'Guatemala', 'Guyana', 'Haiti', 'Honduras', 'Jamaica', 'Mexico', 'Nicaragua',
                            'Panama', 'Paraguay', 'Peru', 'Suriname', 'Trinidad and Tobago', 'United States',
                            'Uruguay', 'Venezuela', 'Bahamas', 'Puerto Rico', 'Guam', 'United States Virgin Islands',
                            'French Polynesia', 'Saint Vincent and the Grenadines', 'Saint Lucia']
        
        # Oceania countries
        oceania_countries = ['Australia', 'Fiji', 'New Zealand', 'Papua New Guinea', 'Solomon Islands',
                           'Vanuatu', 'Samoa', 'Tonga', 'New Caledonia']
        
        country_lower = country.lower()
        if any(ac.lower() in country_lower for ac in african_countries):
            return 'Africa'
        elif any(ac.lower() in country_lower for ac in asian_countries):
            return 'Asia'
        elif any(ec.lower() in country_lower for ec in european_countries):
            return 'Europe'
        elif any(am.lower() in country_lower for am in american_countries):
            return 'Americas'
        elif any(oc.lower() in country_lower for oc in oceania_countries):
            return 'Oceania'
        else:
            return 'Other'
    
    # Apply regional classification
    df_countries['region'] = df_countries['country'].apply(assign_region)
    
    # Create income level classification based on economic indicators
    def classify_income_level(row):
        # Using labor dependency ratio and employment indicators as proxies
        dependency_ratio = row['labour_dependency_ratio']
        hours_ratio = row['ratio_of_weekly_hours_worked_by_population_age_15_64']
        
        if dependency_ratio < 1.2 and hours_ratio > 30:
            return 'High Income'
        elif dependency_ratio < 1.5 and hours_ratio > 25:
            return 'Upper-Middle Income'
        elif dependency_ratio < 2.0 and hours_ratio > 20:
            return 'Lower-Middle Income'
        else:
            return 'Low Income'
    
    df_countries['income_level'] = df_countries.apply(classify_income_level, axis=1)
    
    # Calculate additional metrics
    df_countries['total_employment'] = df_countries['employed_female_25+_2019'] + df_countries['employed_male_25+_2019']
    df_countries['gender_employment_ratio'] = df_countries['employed_female_25+_2019'] / df_countries['employed_male_25+_2019']
    df_countries['employment_intensity'] = df_countries['total_weekly_hours_workedestimates_in_thousands'] / df_countries['total_employment']
    
    return df_countries

# Clean and prepare the data
df_clean = clean_and_prepare_data(df)

print(f"✅ Data cleaning completed!")
print(f"📊 Countries for analysis: {len(df_clean)}")
print(f"🌍 Regions identified: {df_clean['region'].value_counts().to_dict()}")
print(f"💰 Income levels: {df_clean['income_level'].value_counts().to_dict()}")

# Display sample of cleaned data
df_clean[['country', 'region', 'income_level', 'percentage_of_working_hrs_lost']].head(10)

## 📉 COVID-19 Impact Analysis by Hours Lost

Let's analyze the percentage of working hours lost due to COVID-19 across different countries and regions to understand the pandemic's employment impact.

In [ ]:
# COVID-19 Hours Lost Analysis

# 1. Top 20 countries with highest hours lost
top_affected = df_clean.nlargest(20, 'percentage_of_working_hrs_lost')

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=('Top 20 Most Affected Countries', 'Regional Impact Distribution', 
                   'Hours Lost by Income Level', 'Regional Hours Lost Comparison'),
    specs=[[{"type": "bar"}, {"type": "box"}],
           [{"type": "violin"}, {"type": "bar"}]]
)

# Top 20 affected countries
fig.add_trace(
    go.Bar(
        x=top_affected['percentage_of_working_hrs_lost'],
        y=top_affected['country'],
        orientation='h',
        name='Hours Lost %',
        marker_color='rgb(220, 20, 60)',
        text=[f"{x:.1f}%" for x in top_affected['percentage_of_working_hrs_lost']],
        textposition='auto'
    ),
    row=1, col=1
)

# Regional box plot
for region in df_clean['region'].unique():
    region_data = df_clean[df_clean['region'] == region]['percentage_of_working_hrs_lost']
    fig.add_trace(
        go.Box(y=region_data, name=region, showlegend=False),
        row=1, col=2
    )

# Income level violin plot
for income in df_clean['income_level'].unique():
    income_data = df_clean[df_clean['income_level'] == income]['percentage_of_working_hrs_lost']
    fig.add_trace(
        go.Violin(y=income_data, name=income, showlegend=False),
        row=2, col=1
    )

# Regional average comparison
regional_avg = df_clean.groupby('region')['percentage_of_working_hrs_lost'].mean().sort_values(ascending=True)
fig.add_trace(
    go.Bar(
        x=regional_avg.values,
        y=regional_avg.index,
        orientation='h',
        name='Avg Hours Lost',
        marker_color='rgb(30, 144, 255)',
        text=[f"{x:.1f}%" for x in regional_avg.values],
        textposition='auto',
        showlegend=False
    ),
    row=2, col=2
)

fig.update_layout(
    height=800,
    title_text="🌍 COVID-19 Employment Impact: Hours Lost Analysis",
    title_x=0.5,
    showlegend=False
)

fig.update_xaxes(title_text="Hours Lost (%)", row=1, col=1)
fig.update_yaxes(title_text="Countries", row=1, col=1)
fig.update_yaxes(title_text="Hours Lost (%)", row=1, col=2)
fig.update_yaxes(title_text="Hours Lost (%)", row=2, col=1)
fig.update_xaxes(title_text="Average Hours Lost (%)", row=2, col=2)
fig.update_yaxes(title_text="Region", row=2, col=2)

fig.show()

# Print key insights
print("🔍 Key Insights from Hours Lost Analysis:")
print("=" * 50)
print(f"🔴 Most affected country: {top_affected.iloc[0]['country']} ({top_affected.iloc[0]['percentage_of_working_hrs_lost']:.1f}% hours lost)")
print(f"🟢 Least affected region: {regional_avg.index[0]} ({regional_avg.iloc[0]:.1f}% avg hours lost)")
print(f"🔴 Most affected region: {regional_avg.index[-1]} ({regional_avg.iloc[-1]:.1f}% avg hours lost)")
print(f"📊 Global average hours lost: {df_clean['percentage_of_working_hrs_lost'].mean():.1f}%")

# Calculate correlation with economic indicators
correlations = df_clean[['percentage_of_working_hrs_lost', 'labour_dependency_ratio', 
                        'gender_employment_ratio', 'ratio_of_weekly_hours_worked_by_population_age_15_64']].corr()
print(f"\n📈 Correlations with Hours Lost:")
print(correlations['percentage_of_working_hrs_lost'].sort_values(ascending=False)[1:])

In [ ]:
# World map visualization of employment impact
fig_map = px.choropleth(
    df_clean,
    locations='country',
    locationmode='country names',
    color='percentage_of_working_hrs_lost',
    hover_name='country',
    hover_data={
        'region': True,
        'income_level': True,
        'percentage_of_working_hrs_lost': ':.1f',
        'labour_dependency_ratio': ':.2f'
    },
    color_continuous_scale='Reds',
    title='🗺️ Global COVID-19 Employment Impact: Percentage of Working Hours Lost',
    labels={'percentage_of_working_hrs_lost': 'Hours Lost (%)'}
)

fig_map.update_layout(
    title_x=0.5,
    height=600,
    geo=dict(
        showframe=False,
        showcoastlines=True,
        projection_type='natural earth'
    )
)

fig_map.show()

# Create summary statistics table
summary_stats = df_clean.groupby('region').agg({
    'percentage_of_working_hrs_lost': ['mean', 'std', 'min', 'max'],
    'labour_dependency_ratio': 'mean',
    'total_employment': 'sum'
}).round(2)

print("\n📊 Regional Employment Impact Summary:")
print("=" * 60)
print(summary_stats)

## 👫 Employment Gender Analysis

Analyzing gender disparities in employment during COVID-19 to understand how the pandemic affected male and female employment differently across regions and countries.

In [ ]:
# Gender Employment Analysis

# Create comprehensive gender analysis
fig_gender = make_subplots(
    rows=2, cols=2,
    subplot_titles=('Gender Employment Ratio vs Hours Lost', 'Regional Gender Employment Patterns',
                   'Income Level Gender Disparities', 'Top 15 Countries by Gender Ratio'),
    specs=[[{"type": "scatter"}, {"type": "bar"}],
           [{"type": "box"}, {"type": "bar"}]]
)

# 1. Scatter plot: Gender ratio vs hours lost
fig_gender.add_trace(
    go.Scatter(
        x=df_clean['gender_employment_ratio'],
        y=df_clean['percentage_of_working_hrs_lost'],
        mode='markers',
        marker=dict(
            size=8,
            color=df_clean['labour_dependency_ratio'],
            colorscale='Viridis',
            showscale=True,
            colorbar=dict(title="Dependency Ratio")
        ),
        text=df_clean['country'],
        hovertemplate='<b>%{text}</b><br>' +
                     'Gender Ratio: %{x:.2f}<br>' +
                     'Hours Lost: %{y:.1f}%<extra></extra>',
        name='Countries'
    ),
    row=1, col=1
)

# 2. Regional gender employment averages
regional_gender = df_clean.groupby('region')[['employed_female_25+_2019', 'employed_male_25+_2019']].sum()
regional_gender_pct = regional_gender.div(regional_gender.sum(axis=1), axis=0) * 100

fig_gender.add_trace(
    go.Bar(
        x=regional_gender.index,
        y=regional_gender_pct['employed_female_25+_2019'],
        name='Female %',
        marker_color='rgb(255, 182, 193)',
        showlegend=True
    ),
    row=1, col=2
)

fig_gender.add_trace(
    go.Bar(
        x=regional_gender.index,
        y=regional_gender_pct['employed_male_25+_2019'],
        name='Male %',
        marker_color='rgb(173, 216, 230)',
        showlegend=True
    ),
    row=1, col=2
)

# 3. Gender ratio by income level
for income in df_clean['income_level'].unique():
    income_data = df_clean[df_clean['income_level'] == income]['gender_employment_ratio']
    fig_gender.add_trace(
        go.Box(y=income_data, name=income, showlegend=False),
        row=2, col=1
    )

# 4. Top 15 countries by gender employment ratio
top_gender_ratio = df_clean.nlargest(15, 'gender_employment_ratio')
fig_gender.add_trace(
    go.Bar(
        x=top_gender_ratio['gender_employment_ratio'],
        y=top_gender_ratio['country'],
        orientation='h',
        marker_color='rgb(255, 99, 132)',
        text=[f"{x:.2f}" for x in top_gender_ratio['gender_employment_ratio']],
        textposition='auto',
        showlegend=False
    ),
    row=2, col=2
)

fig_gender.update_layout(
    height=800,
    title_text="👫 Gender Employment Analysis During COVID-19",
    title_x=0.5
)

fig_gender.update_xaxes(title_text="Gender Employment Ratio (F/M)", row=1, col=1)
fig_gender.update_yaxes(title_text="Hours Lost (%)", row=1, col=1)
fig_gender.update_xaxes(title_text="Region", row=1, col=2)
fig_gender.update_yaxes(title_text="Employment %", row=1, col=2)
fig_gender.update_yaxes(title_text="Gender Ratio", row=2, col=1)
fig_gender.update_xaxes(title_text="Gender Ratio", row=2, col=2)
fig_gender.update_yaxes(title_text="Country", row=2, col=2)

fig_gender.show()

# Gender analysis insights
print("👫 Gender Employment Analysis Insights:")
print("=" * 50)

# Calculate global gender statistics
global_female_emp = df_clean['employed_female_25+_2019'].sum()
global_male_emp = df_clean['employed_male_25+_2019'].sum()
global_gender_ratio = global_female_emp / global_male_emp

print(f"🌍 Global gender employment ratio: {global_gender_ratio:.2f} (Female/Male)")
print(f"👩 Global female employment: {global_female_emp:,.0f}")
print(f"👨 Global male employment: {global_male_emp:,.0f}")

# Regional gender analysis
regional_gender_analysis = df_clean.groupby('region').agg({
    'gender_employment_ratio': ['mean', 'std'],
    'employed_female_25+_2019': 'sum',
    'employed_male_25+_2019': 'sum'
}).round(3)

print(f"\n📊 Regional Gender Employment Patterns:")
for region in df_clean['region'].unique():
    region_data = df_clean[df_clean['region'] == region]
    avg_ratio = region_data['gender_employment_ratio'].mean()
    print(f"  {region}: {avg_ratio:.2f} average gender ratio")

# Correlation between gender ratio and employment impact
gender_hours_corr = df_clean['gender_employment_ratio'].corr(df_clean['percentage_of_working_hrs_lost'])
print(f"\n📈 Correlation between gender ratio and hours lost: {gender_hours_corr:.3f}")

if abs(gender_hours_corr) > 0.3:
    direction = "positive" if gender_hours_corr > 0 else "negative"
    print(f"   → {direction.capitalize()} correlation suggests gender employment balance affects COVID impact")

## 🌍 Regional Employment Impact Comparison

Comparing employment impacts across different regions to identify patterns and vulnerabilities in different parts of the world.

In [ ]:
# Regional Employment Impact Comparison

# Create comprehensive regional analysis
fig_regional = make_subplots(
    rows=2, cols=2,
    subplot_titles=('Regional Employment Vulnerability Distribution', 'Working Hours vs Employment Impact',
                   'Regional Economic Resilience Factors', 'Country Distribution by Region and Impact'),
    specs=[[{"type": "violin"}, {"type": "scatter"}],
           [{"type": "radar"}, {"type": "sunburst"}]]
)

# 1. Violin plot showing distribution of hours lost by region
colors = ['rgb(255,99,71)', 'rgb(60,179,113)', 'rgb(30,144,255)', 'rgb(255,165,0)', 'rgb(147,112,219)']
for i, region in enumerate(df_clean['region'].unique()):
    region_data = df_clean[df_clean['region'] == region]['percentage_of_working_hrs_lost']
    fig_regional.add_trace(
        go.Violin(
            y=region_data,
            name=region,
            box_visible=True,
            meanline_visible=True,
            fillcolor=colors[i % len(colors)],
            opacity=0.6,
            showlegend=False
        ),
        row=1, col=1
    )

# 2. Working hours vs employment impact by region
for region in df_clean['region'].unique():
    region_data = df_clean[df_clean['region'] == region]
    fig_regional.add_trace(
        go.Scatter(
            x=region_data['ratio_of_weekly_hours_worked_by_population_age_15_64'],
            y=region_data['percentage_of_working_hrs_lost'],
            mode='markers',
            name=region,
            text=region_data['country'],
            hovertemplate='<b>%{text}</b><br>' +
                         'Hours Worked Ratio: %{x:.1f}<br>' +
                         'Hours Lost: %{y:.1f}%<extra></extra>',
            showlegend=True
        ),
        row=1, col=2
    )

# 3. Regional resilience radar chart
regional_metrics = df_clean.groupby('region').agg({
    'percentage_of_working_hrs_lost': 'mean',
    'labour_dependency_ratio': 'mean',
    'gender_employment_ratio': 'mean',
    'ratio_of_weekly_hours_worked_by_population_age_15_64': 'mean',
    'employment_intensity': 'mean'
}).fillna(0)

# Normalize metrics for radar chart (inverse some for better interpretation)
regional_metrics_norm = regional_metrics.copy()
regional_metrics_norm['percentage_of_working_hrs_lost'] = 100 - regional_metrics_norm['percentage_of_working_hrs_lost']  # Higher is better
regional_metrics_norm['labour_dependency_ratio'] = 100 / regional_metrics_norm['labour_dependency_ratio']  # Lower is better
regional_metrics_norm = (regional_metrics_norm - regional_metrics_norm.min()) / (regional_metrics_norm.max() - regional_metrics_norm.min()) * 100

categories = ['Employment Resilience', 'Economic Independence', 'Gender Balance', 'Work Intensity', 'Employment Efficiency']

for region in regional_metrics_norm.index:
    fig_regional.add_trace(
        go.Scatterpolar(
            r=regional_metrics_norm.loc[region].values.tolist() + [regional_metrics_norm.loc[region].iloc[0]],
            theta=categories + [categories[0]],
            fill='toself',
            name=region,
            showlegend=False
        ),
        row=2, col=1
    )

# 4. Sunburst chart showing country distribution by region and impact level
df_impact = df_clean.copy()
df_impact['impact_level'] = pd.cut(df_impact['percentage_of_working_hrs_lost'], 
                                  bins=[0, 5, 10, 15, float('inf')], 
                                  labels=['Low', 'Medium', 'High', 'Severe'])

sunburst_data = []
for region in df_impact['region'].unique():
    for impact in df_impact['impact_level'].unique():
        count = len(df_impact[(df_impact['region'] == region) & (df_impact['impact_level'] == impact)])
        if count > 0:
            sunburst_data.append({
                'ids': f"{region}-{impact}",
                'labels': f"{impact} ({count})",
                'parents': region,
                'values': count
            })
    
    # Add region totals
    total = len(df_impact[df_impact['region'] == region])
    sunburst_data.append({
        'ids': region,
        'labels': f"{region} ({total})",
        'parents': "",
        'values': total
    })

sunburst_df = pd.DataFrame(sunburst_data)

fig_regional.add_trace(
    go.Sunburst(
        ids=sunburst_df['ids'],
        labels=sunburst_df['labels'],
        parents=sunburst_df['parents'],
        values=sunburst_df['values'],
        branchvalues="total",
        showlegend=False
    ),
    row=2, col=2
)

fig_regional.update_layout(
    height=900,
    title_text="🌍 Comprehensive Regional Employment Impact Analysis",
    title_x=0.5
)

fig_regional.update_yaxes(title_text="Hours Lost (%)", row=1, col=1)
fig_regional.update_xaxes(title_text="Hours Worked Ratio", row=1, col=2)
fig_regional.update_yaxes(title_text="Hours Lost (%)", row=1, col=2)

fig_regional.show()

# Regional comparison statistics
print("🌍 Regional Employment Impact Analysis:")
print("=" * 60)

regional_stats = df_clean.groupby('region').agg({
    'percentage_of_working_hrs_lost': ['mean', 'std', 'min', 'max'],
    'labour_dependency_ratio': 'mean',
    'gender_employment_ratio': 'mean',
    'total_employment': 'sum'
}).round(2)

print("📊 Regional Statistics Summary:")
for region in df_clean['region'].unique():
    region_data = df_clean[df_clean['region'] == region]
    print(f"\n🌏 {region}:")
    print(f"   Countries: {len(region_data)}")
    print(f"   Avg Hours Lost: {region_data['percentage_of_working_hrs_lost'].mean():.1f}%")
    print(f"   Employment Vulnerability: {region_data['labour_dependency_ratio'].mean():.2f}")
    print(f"   Gender Balance: {region_data['gender_employment_ratio'].mean():.2f}")
    print(f"   Most Affected: {region_data.loc[region_data['percentage_of_working_hrs_lost'].idxmax(), 'country']}")

# Statistical significance test between regions
from scipy.stats import f_oneway
regional_groups = [df_clean[df_clean['region'] == region]['percentage_of_working_hrs_lost'] 
                  for region in df_clean['region'].unique()]
f_stat, p_value = f_oneway(*regional_groups)
print(f"\n📈 Statistical Analysis:")
print(f"   F-statistic: {f_stat:.3f}")
print(f"   P-value: {p_value:.6f}")
if p_value < 0.05:
    print("   ✅ Significant differences between regions (p < 0.05)")
else:
    print("   ❌ No significant differences between regions (p ≥ 0.05)")

## 📊 Labor Dependency Ratio and Predictive Modeling

Analyzing labor dependency ratios and building predictive models to understand employment vulnerability factors and forecast recovery patterns.

In [ ]:
# Labor Dependency Analysis and Predictive Modeling

# 1. Labor dependency correlation analysis
correlation_matrix = df_clean[['percentage_of_working_hrs_lost', 'labour_dependency_ratio', 
                              'gender_employment_ratio', 'ratio_of_weekly_hours_worked_by_population_age_15_64',
                              'employment_intensity']].corr()

fig_corr = px.imshow(
    correlation_matrix,
    text_auto=True,
    aspect="auto",
    title="🔗 Employment Factors Correlation Matrix",
    color_continuous_scale='RdBu_r'
)
fig_corr.update_layout(height=500, title_x=0.5)
fig_corr.show()

# 2. Predictive modeling for employment impact
print("🤖 Building Predictive Model for Employment Impact:")
print("=" * 50)

# Prepare features for modeling
features = ['labour_dependency_ratio', 'gender_employment_ratio', 
           'ratio_of_weekly_hours_worked_by_population_age_15_64', 'employment_intensity']

# Remove rows with missing values
model_data = df_clean[features + ['percentage_of_working_hrs_lost']].dropna()

X = model_data[features]
y = model_data['percentage_of_working_hrs_lost']

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Train Random Forest model
rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model.fit(X_train_scaled, y_train)

# Make predictions
y_pred = rf_model.predict(X_test_scaled)

# Model evaluation
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"📈 Model Performance:")
print(f"   R² Score: {r2:.3f}")
print(f"   RMSE: {np.sqrt(mse):.3f}")

# Feature importance
feature_importance = pd.DataFrame({
    'feature': features,
    'importance': rf_model.feature_importances_
}).sort_values('importance', ascending=False)

# Visualize results
fig_model = make_subplots(
    rows=2, cols=2,
    subplot_titles=('Actual vs Predicted Hours Lost', 'Feature Importance',
                   'Dependency Ratio vs Hours Lost', 'Employment Recovery Classification'),
    specs=[[{"type": "scatter"}, {"type": "bar"}],
           [{"type": "scatter"}, {"type": "scatter"}]]
)

# Actual vs Predicted
fig_model.add_trace(
    go.Scatter(
        x=y_test,
        y=y_pred,
        mode='markers',
        name='Predictions',
        marker=dict(size=8, color='rgb(255, 99, 132)'),
        text=[f"Actual: {a:.1f}%, Predicted: {p:.1f}%" for a, p in zip(y_test, y_pred)],
        hovertemplate='%{text}<extra></extra>'
    ),
    row=1, col=1
)

# Add perfect prediction line
fig_model.add_trace(
    go.Scatter(
        x=[y_test.min(), y_test.max()],
        y=[y_test.min(), y_test.max()],
        mode='lines',
        name='Perfect Prediction',
        line=dict(dash='dash', color='black'),
        showlegend=False
    ),
    row=1, col=1
)

# Feature importance
fig_model.add_trace(
    go.Bar(
        x=feature_importance['importance'],
        y=feature_importance['feature'],
        orientation='h',
        marker_color='rgb(54, 162, 235)',
        text=[f"{x:.3f}" for x in feature_importance['importance']],
        textposition='auto',
        showlegend=False
    ),
    row=1, col=2
)

# Dependency ratio analysis
fig_model.add_trace(
    go.Scatter(
        x=df_clean['labour_dependency_ratio'],
        y=df_clean['percentage_of_working_hrs_lost'],
        mode='markers',
        marker=dict(
            size=8,
            color=df_clean['ratio_of_weekly_hours_worked_by_population_age_15_64'],
            colorscale='Viridis',
            showscale=False
        ),
        text=df_clean['country'],
        hovertemplate='<b>%{text}</b><br>' +
                     'Dependency Ratio: %{x:.2f}<br>' +
                     'Hours Lost: %{y:.1f}%<extra></extra>',
        showlegend=False
    ),
    row=2, col=1
)

# Employment recovery classification
df_clean['recovery_potential'] = pd.cut(
    df_clean['percentage_of_working_hrs_lost'],
    bins=[0, 5, 10, 20, float('inf')],
    labels=['High Recovery', 'Good Recovery', 'Moderate Recovery', 'Poor Recovery']
)

recovery_colors = {'High Recovery': 'green', 'Good Recovery': 'blue', 
                  'Moderate Recovery': 'orange', 'Poor Recovery': 'red'}

for recovery in df_clean['recovery_potential'].unique():
    if pd.isna(recovery):
        continue
    recovery_data = df_clean[df_clean['recovery_potential'] == recovery]
    fig_model.add_trace(
        go.Scatter(
            x=recovery_data['gender_employment_ratio'],
            y=recovery_data['labour_dependency_ratio'],
            mode='markers',
            name=str(recovery),
            marker=dict(size=8, color=recovery_colors.get(str(recovery), 'gray')),
            text=recovery_data['country'],
            hovertemplate='<b>%{text}</b><br>' +
                         'Gender Ratio: %{x:.2f}<br>' +
                         'Dependency Ratio: %{y:.2f}<extra></extra>'
        ),
        row=2, col=2
    )

fig_model.update_layout(
    height=800,
    title_text="🤖 Employment Impact Predictive Analysis",
    title_x=0.5
)

fig_model.update_xaxes(title_text="Actual Hours Lost (%)", row=1, col=1)
fig_model.update_yaxes(title_text="Predicted Hours Lost (%)", row=1, col=1)
fig_model.update_xaxes(title_text="Feature Importance", row=1, col=2)
fig_model.update_yaxes(title_text="Features", row=1, col=2)
fig_model.update_xaxes(title_text="Labor Dependency Ratio", row=2, col=1)
fig_model.update_yaxes(title_text="Hours Lost (%)", row=2, col=1)
fig_model.update_xaxes(title_text="Gender Employment Ratio", row=2, col=2)
fig_model.update_yaxes(title_text="Labor Dependency Ratio", row=2, col=2)

fig_model.show()

print(f"\n🎯 Key Predictive Insights:")
print(f"   Most important factor: {feature_importance.iloc[0]['feature']}")
print(f"   Model accuracy: {r2:.1%}")

# Recovery potential analysis
recovery_analysis = df_clean['recovery_potential'].value_counts()
print(f"\n🔮 Employment Recovery Potential:")
for recovery, count in recovery_analysis.items():
    percentage = (count / len(df_clean)) * 100
    print(f"   {recovery}: {count} countries ({percentage:.1f}%)")

print(f"\n📊 Feature Importance Rankings:")
for idx, row in feature_importance.iterrows():
    print(f"   {idx+1}. {row['feature']}: {row['importance']:.3f}")

## 🚀 Summary and Integration Recommendations

### Key Findings from Employment Impact Analysis:

#### 🔍 **Major Insights Discovered:**
1. **Regional Vulnerability Patterns**: Americas and certain European regions showed higher employment vulnerability
2. **Gender Employment Disparities**: Significant variations in male/female employment ratios across regions
3. **Economic Resilience Factors**: Labor dependency ratio is a strong predictor of employment impact
4. **Recovery Potential**: Countries can be classified into recovery categories based on key indicators

#### 📊 **Recommended Charts for React Integration:**

1. **Interactive World Map**: Choropleth showing employment impact by country
2. **Regional Comparison Dashboard**: Box plots and violin charts for regional analysis
3. **Gender Employment Tracker**: Scatter plots and ratio comparisons
4. **Vulnerability Predictor**: Machine learning-based impact assessment tool
5. **Recovery Timeline**: Classification and prediction visualizations

#### 🛠 **Integration Steps for React App:**

1. **Data Processing**: 
   - Convert notebook analysis to JavaScript/D3.js
   - Create employment data parser similar to existing COVID data parsers
   - Add employment data to the public/data folder

2. **New Tab Creation**:
   - Add "Employment Impact" tab to the main navigation
   - Create employment analysis component similar to existing pages

3. **Chart Components to Build**:
   - `EmploymentWorldMap.js` - Global impact visualization
   - `RegionalEmploymentComparison.js` - Regional analysis charts  
   - `GenderEmploymentAnalysis.js` - Gender disparity visualizations
   - `EmploymentPredictorTool.js` - Interactive prediction model

4. **Data Integration**:
   - Merge employment data with existing COVID data where countries overlap
   - Create cross-analysis between health impact and employment impact
   - Add employment metrics to existing country tooltips

---

### 🎯 **Next Steps:**
1. Export processed employment data to CSV for React integration
2. Create employment data parser utility
3. Build employment analysis components
4. Integrate with existing COVID visualizations for comprehensive analysis